# Build a validated regression set

Python 3.12 · offline · estimated time: 18 minutes

Run each cell in order. The final cell persists `eval_cases.jsonl`, which EP-02 will consume.

In [1]:
# Edit this criterion only if you also relabel every case below.
criterion = (
    'Regression present: the reply says a customer is eligible for a refund '
    'when the supplied policy says the request is outside the refund window.'
)
label_meaning = {1: 'regression present', 0: 'acceptable under this criterion'}
print('CHECK 1 — criterion:', criterion)
print('CHECK 1 — label mapping: 1 = regression present; 0 = acceptable')

CHECK 1 — criterion: Regression present: the reply says a customer is eligible for a refund when the supplied policy says the request is outside the refund window.
CHECK 1 — label mapping: 1 = regression present; 0 = acceptable


In [2]:
# Edit these records while preserving the four-field interface.
records = [
    {
        'case_id': 'refund-window-001',
        'input': 'Policy: refunds are available within 30 days. Request: day 45.',
        'candidate_output': 'You are eligible for a full refund.',
        'human_label': 1,
    },
    {
        'case_id': 'refund-window-002',
        'input': 'Policy: refunds are available within 30 days. Request: day 45.',
        'candidate_output': 'The 30-day refund window has passed, so this purchase is not eligible.',
        'human_label': 0,
    },
    {
        'case_id': 'refund-window-003',
        'input': 'Policy: refunds are available within 30 days. Request: day 33.',
        'candidate_output': 'Because you are only three days late, we will process your refund.',
        'human_label': 1,
    },
    {
        'case_id': 'refund-window-004',
        'input': 'Policy: refunds are available within 30 days. Request: day 12.',
        'candidate_output': 'Your request is within 30 days and is eligible for a refund.',
        'human_label': 0,
    },
]
print(f'CHECK 2 — created {len(records)} cases')
print('CHECK 2 — preview:', [(row['case_id'], row['human_label']) for row in records])

CHECK 2 — created 4 cases
CHECK 2 — preview: [('refund-window-001', 1), ('refund-window-002', 0), ('refund-window-003', 1), ('refund-window-004', 0)]


In [3]:
from collections import Counter

REQUIRED = {'case_id': str, 'input': str, 'candidate_output': str, 'human_label': int}

def validate(rows):
    errors, seen = [], set()
    for number, row in enumerate(rows, start=1):
        for field, expected_type in REQUIRED.items():
            if field not in row:
                errors.append(f'row {number}: missing {field}')
            elif type(row[field]) is not expected_type:
                errors.append(f'row {number}: {field} must be {expected_type.__name__}')
            elif expected_type is str and not row[field].strip():
                errors.append(f'row {number}: {field} must not be empty')
        case_id = row.get('case_id')
        if isinstance(case_id, str) and case_id in seen:
            errors.append(f'row {number}: duplicate case_id {case_id!r}')
        seen.add(case_id)
        if type(row.get('human_label')) is int and row['human_label'] not in (0, 1):
            errors.append(f'row {number}: human_label must be 0 or 1')
    if errors:
        raise ValueError('\n'.join(errors))

validate(records)
print('CHECK 3 — schema, types, labels, and unique IDs: PASS')

CHECK 3 — schema, types, labels, and unique IDs: PASS


In [4]:
counts = Counter(row['human_label'] for row in records)
if not (counts[0] and counts[1]):
    raise ValueError('both classes are required: add at least one label 0 and one label 1')
print(f'CHECK 4 — class counts: label 0 = {counts[0]}, label 1 = {counts[1]}')
print('CHECK 4 — both-class course rule: PASS')

CHECK 4 — class counts: label 0 = 2, label 1 = 2
CHECK 4 — both-class course rule: PASS


In [5]:
import json
from pathlib import Path

artifact_path = (Path('build/lesson-01/eval_cases.jsonl') if Path('build/lesson-01').is_dir() else Path('eval_cases.jsonl'))
artifact_path.parent.mkdir(parents=True, exist_ok=True)
artifact_path.write_text(''.join(json.dumps(row) + '\n' for row in records), encoding='utf-8')
reloaded = [json.loads(line) for line in artifact_path.read_text(encoding='utf-8').splitlines()]
assert reloaded == records, 'read-back records differ from in-memory records'
print(f'CHECK 5 — wrote and reloaded {len(reloaded)} records: PASS')
print('FINAL PASS — artifact ready for EP-02: build/lesson-01/eval_cases.jsonl')

CHECK 5 — wrote and reloaded 4 records: PASS
FINAL PASS — artifact ready for EP-02: build/lesson-01/eval_cases.jsonl
